# Gradio Front-End for the Airbnb Price Serverless Endpoint

This notebook builds a **Gradio front-end UI** that invokes the SageMaker Serverless Endpoint
created in `03_production_pipeline.ipynb` — the *fair advertised nightly rate* regressor.

Default example configuration:

| Item | Value |
|---|---|
| Team | `team14` |
| Student/Profile | `s1402` |
| Region | `ap-southeast-1` |
| Endpoint name | `iti113-team14-airbnb-price-slsv2` |
| Task | Regression — continuous nightly `price`, in the listing's local currency |
| Contract | 27 raw listing fields (`SERVING_COLUMNS`), JSON |

This Gradio app does **not train a model**, and it does **not preprocess anything**. It collects
the 27 raw listing fields, formats them as a JSON object, sends them to the deployed SageMaker
Serverless Endpoint, and displays the prediction response.

Everything that turns a raw listing into features — distance from the city centre, amenity flags,
neighbourhood frequency, host tenure, scaling, one-hot encoding — happens **inside the endpoint**,
in the same fitted `Pipeline` object that training serialized. That is the train–serve consistency
guarantee proved in Notebook 03 Step 4, and this front-end is deliberately built so that it cannot
undermine it:

```text
Gradio UI  ──JSON──▶  inference.py  ──▶  ListingFeatureEngineer  ──▶  AirbnbPreprocessor
                                    ──▶  TransformedTargetRegressor(HistGradientBoosting)
                                    ──▶  suggested nightly price, local currency
```

## 1. Install Required Package

Run this once in SageMaker Studio. If Gradio is already installed, this cell will complete quickly.

In [1]:
# !pip install -q "starlette<1" "fastapi<1" --upgrade
# !pip install -q gradio

## 2. Configure Team, User, Region, and Check for Endpoint

For the class demo, the default endpoint is the Team14 endpoint:

```text
iti113-team14-airbnb-price-slsv2
```

Students must replace these values with their own team details. The naming convention comes
straight from Notebook 03 Step 0:

```python
ENDPOINT_NAME = f"iti113-{TEAM_ID}-airbnb-price-slsv2"
```

This code searches for serverless endpoints created by your own team.

In [2]:
import boto3
import json
import gradio as gr
from datetime import datetime

REGION = "ap-southeast-1"

# -------------------------------------------------------------------
# TODO: Students should change these values for their own team.
# Example:
# TEAM_ID = "team01"
# STUDENT_ID = "s101"
# ENDPOINT_NAME = "iti113-team01-airbnb-price-slsv2"
# -------------------------------------------------------------------
TEAM_ID       = "team14"
STUDENT_ID    = "s1402"
COURSE        = "ITI113"
PROJECT_NAME  = "airbnb-listings"
ENDPOINT_NAME = f"iti113-{TEAM_ID}-airbnb-price-sls"

# Where Notebook 01/03 published the fitted encoder state (used in Section 5)
BUCKET           = "nyp-26s1-iti113"
PREFIX           = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"
PROCESSED_PREFIX = f"{PREFIX}/processed"
ARTIFACTS_PREFIX = f"{PREFIX}/artifacts"

sts = boto3.client("sts", region_name=REGION)

print("Region:", REGION)
print("Team ID:", TEAM_ID)
print("Student ID:", STUDENT_ID)
print("AWS identity:", sts.get_caller_identity()["Arn"])

sm      = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)
s3      = boto3.client("s3", region_name=REGION)

response = sm.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=100
)

team_endpoints = []

for ep in response["Endpoints"]:
    endpoint_name = ep["EndpointName"]

    # Only show endpoints that follow the team naming convention
    if TEAM_ID in endpoint_name.lower():
        team_endpoints.append(ep)

print(f"\nEndpoints found for {TEAM_ID}: {len(team_endpoints)}\n")

for ep in team_endpoints:
    marker = "   <-- this notebook" if ep["EndpointName"] == ENDPOINT_NAME else ""
    print("Endpoint name:", ep["EndpointName"], marker)
    print("Status:", ep["EndpointStatus"])
    print("Creation time:", ep["CreationTime"])
    print("Last modified:", ep["LastModifiedTime"])
    print("-" * 80)

if ENDPOINT_NAME not in {ep["EndpointName"] for ep in team_endpoints}:
    print(f"WARNING: {ENDPOINT_NAME} is not in the list above.")
    print("Set ENDPOINT_NAME to one of the names shown, or run Notebook 03 Step 8 to deploy it.")

Region: ap-southeast-1
Team ID: team14
Student ID: s1402
AWS identity: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team14/SageMaker

Endpoints found for team14: 1

Endpoint name: iti113-team14-airbnb-price-sls    <-- this notebook
Status: InService
Creation time: 2026-08-18 02:22:19.663000+00:00
Last modified: 2026-08-18 02:24:47.720000+00:00
--------------------------------------------------------------------------------


## 3. Confirm That the Serverless Endpoint(s) Is Active

The endpoint must show `InService` before it can be invoked.

Note the loop variable below is `name`, not `ENDPOINT_NAME`. Reassigning `ENDPOINT_NAME` inside a
loop would leave it pointing at whichever endpoint happened to come last, and the Gradio UI further
down would then quietly talk to the wrong model.

In [3]:
try:
    for ep in team_endpoints:
        name = ep["EndpointName"]
        endpoint_desc = sm.describe_endpoint(EndpointName=name)
        print("\nEndpoint name:", endpoint_desc["EndpointName"])
        print("Status:", endpoint_desc["EndpointStatus"])
        print("Creation time:", endpoint_desc["CreationTime"])
        print("Last modified:", endpoint_desc["LastModifiedTime"])
except Exception as e:
    print("Unable to describe endpoint.")
    print("Check that ENDPOINT_NAME is correct and that your role has permission to access it.")
    print("Error:", e)

# The one this notebook will actually invoke
try:
    status = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
except Exception:
    status = "NotFound"

print("\n" + "=" * 80)
print(f"Target endpoint: {ENDPOINT_NAME} -> {status}")
if status == "Creating":
    print("Still provisioning. Deployment takes 3-5 minutes; re-run this cell.")
elif status == "NotFound":
    print("No endpoint with that name in this region. Run Notebook 03 Step 8 first.")


Endpoint name: iti113-team14-airbnb-price-sls
Status: InService
Creation time: 2026-08-18 02:22:19.663000+00:00
Last modified: 2026-08-18 02:24:47.720000+00:00

Target endpoint: iti113-team14-airbnb-price-sls -> InService


## 4. Inspect the Endpoint Model Artifact (Optional)

This cell shows which SageMaker model, endpoint configuration and model package are behind the
endpoint. This is how it connect the deployed endpoint back to the `model.tar.gz` produced by
Notebook 03 — the lineage check a reviewer performs before trusting a prediction.

In [4]:
for ep in team_endpoints:
    name = ep["EndpointName"]

    print("=" * 100)
    print("Endpoint name:", name)

    try:
        # 1. Describe endpoint
        endpoint_desc = sm.describe_endpoint(EndpointName=name)

        endpoint_config_name = endpoint_desc["EndpointConfigName"]

        print("Endpoint status:", endpoint_desc["EndpointStatus"])
        print("Endpoint config:", endpoint_config_name)

        # 2. Describe endpoint config
        endpoint_config = sm.describe_endpoint_config(
            EndpointConfigName=endpoint_config_name
        )

        production_variants = endpoint_config.get("ProductionVariants", [])

        if not production_variants:
            print("No production variants found.")
            continue

        # 3. Loop through each production variant
        for variant in production_variants:
            print("-" * 80)

            variant_name = variant.get("VariantName")
            model_name = variant.get("ModelName")

            print("Variant name:", variant_name)
            print("Model name:", model_name)

            # Check whether endpoint is serverless or real-time instance
            if "ServerlessConfig" in variant:
                serverless_config = variant["ServerlessConfig"]
                print("Endpoint type: Serverless")
                print("Memory size:", serverless_config.get("MemorySizeInMB"), "MB")
                print("Max concurrency:", serverless_config.get("MaxConcurrency"))
            else:
                print("Endpoint type: Real-time instance")
                print("Instance type:", variant.get("InstanceType"))
                print("Initial instance count:", variant.get("InitialInstanceCount"))

            if not model_name:
                print("No model name found for this variant.")
                continue

            # 4. Describe model
            model_desc = sm.describe_model(ModelName=model_name)

            # 5. Handle PrimaryContainer or Containers
            containers = model_desc.get("Containers")
            if containers is None and "PrimaryContainer" in model_desc:
                containers = [model_desc["PrimaryContainer"]]
                print("Container format: PrimaryContainer")
            else:
                print("Container format: Containers")

            for i, container in enumerate(containers or [], start=1):
                print(f"Container {i} image:", container.get("Image", "Not shown"))
                print(f"Container {i} model artifact S3 URI:", container.get("ModelDataUrl", "Not shown"))

                if "ModelPackageName" in container:
                    model_package_arn = container["ModelPackageName"]
                    print(f"Container {i} model package ARN:", model_package_arn)

                    try:
                        package_desc = sm.describe_model_package(
                            ModelPackageName=model_package_arn
                        )

                        inference_spec = package_desc.get("InferenceSpecification", {})
                        package_containers = inference_spec.get("Containers", [])

                        print("Model package approval status:",
                              package_desc.get("ModelApprovalStatus", "Not shown"))

                        for j, pkg_container in enumerate(package_containers, start=1):
                            print(f"Package container {j} image:", pkg_container.get("Image", "Not shown"))
                            print(f"Package container {j} model artifact S3 URI:",
                                  pkg_container.get("ModelDataUrl", "Not shown"))

                    except Exception as e:
                        print("Unable to inspect model package details.")
                        print("Error:", e)
    except Exception as e:
        print("Unable to inspect this endpoint's model details.")
        print("Error:", e)

print("=" * 100)
print("Endpoint inspection completed.")

Endpoint name: iti113-team14-airbnb-price-sls
Endpoint status: InService
Endpoint config: iti113-team14-airbnb-price-sls


--------------------------------------------------------------------------------
Variant name: AllTraffic
Model name: iti113-team14-airbnb-price-models-2026-08-14-02-59-09-763
Endpoint type: Serverless
Memory size: 3072 MB
Max concurrency: 5


Container format: Containers
Container 1 image: Not shown
Container 1 model artifact S3 URI: Not shown
Container 1 model package ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/iti113-team14-airbnb-price-models/1
Model package approval status: Approved
Package container 1 image: 121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3
Package container 1 model artifact S3 URI: s3://sagemaker-ap-southeast-1-044528205969/sagemaker-scikit-learn-2026-08-14-02-46-26-635/pipelines-atsx2v2jchha-RegisterModel-Repack-LNleag3sJc/output/model.tar.gz
Endpoint inspection completed.


## Inspecting the Deployed Endpoint

The endpoint is connected to a SageMaker Model, and the model points to a Model Registry package
rather than directly exposing the container image and model artifact.

For this Team14 example:

- Endpoint: `iti113-team14-airbnb-price-slsv2`
- Endpoint type: Serverless — 3072 MB, `max_concurrency=5`, scale-to-zero
- Model package group: `iti113-team14-airbnb-price-models`
- Model package status: `Approved` (Notebook 03 Step 8, a deliberate human act)
- Container image: SageMaker Scikit-learn container
- Model artifact: `model.tar.gz` in S3, containing `model_pipeline.joblib`, `inference.py` and
  `pipeline_lib.py`

Alongside it in the MLflow registry:

- Registered model: `iti113-team14-airbnb-price-regressor` version 7, alias `champion`
- Data version: `sha256:097b0bbfea3c`
- Quality gate: `test_r2_log` 0.8708 ≥ 0.65 — **PASSED**

This confirms the endpoint is not merely created, but connected to an approved model package and a
saved model artifact whose lineage traces back to a specific dataset version and a gated run.

For the Gradio UI, these three values are needed:

- `REGION = "ap-southeast-1"`
- `TEAM_ID = "team14"`
- `ENDPOINT_NAME = "iti113-team14-airbnb-price-slsv2"` etc

### Check the endpoint invocation used previously from `03_production_pipeline.ipynb`

Notebook 03 Step 8.2 smoke-tests the same handler chain locally, then Step 8.3 invokes the live
endpoint with exactly the payload format used below.

## 5. Test Endpoint Invocation with boto3

Before building the Gradio UI, test the endpoint directly using `boto3`.

The endpoint expects input in **JSON format**, not CSV. Each input field is sent using its raw
feature name. A request is either one listing object or a list of them.

The deployed model bundle contains the trained pipeline *and* all preprocessing state, so the
endpoint accepts the **27 raw listing fields** of the `SERVING_COLUMNS` contract:

```text
city, latitude, longitude, neighbourhood, property_type, room_type,
accommodates, bedrooms, amenities, minimum_nights, maximum_nights,
instant_bookable, host_since, host_is_superhost, host_has_profile_pic,
host_identity_verified, host_total_listings_count, host_response_time,
host_response_rate, host_acceptance_rate,
review_scores_rating, review_scores_accuracy, review_scores_cleanliness,
review_scores_checkin, review_scores_communication, review_scores_location,
review_scores_value
```

The endpoint's `inference.py` performs all preprocessing internally:

```text
distance from city centre + coordinate coarsening
amenity parsing, amenity_count, rare_amenity_count
neighbourhood frequency lookup
host tenure, missing-value flags
property_type -> one of 7 governed property groups
numeric scaling, one-hot encoding, feature ordering (88 columns)
prediction, then expm1 back to price units
```

**Every one of the 27 keys must be present.** `input_fn` raises on a missing key. `null` is allowed
only for the fields the pipeline explicitly imputes:

| Field(s) | What `null` does |
|---|---|
| `bedrooms` | sets `bedrooms_missing`, fills from `ceil(accommodates / 2)` |
| `host_since` | `host_tenure_days` imputed to the fitted median |
| `host_response_rate`, `host_acceptance_rate` | sets the matching `_missing` flag, imputes the median |
| `host_response_time` | becomes the `"no history"` level |
| `neighbourhood` | becomes `"unknown"`, frequency `0.0` |
| the seven `review_scores_*` | sets `has_review_scores = 0`, imputes the medians |

Anything else sent as `null` produces a `ModelError`: the pipeline neither coerces nor imputes it,
and the endpoint fails its "no NaNs after preprocessing" assertion.

The response is a list, one record per listing:

```json
[{"suggested_nightly_price": 70.9, "currency": "EUR", "city": "Paris", "log_price": 4.2752}]
```

**Important:** the endpoint expects `ContentType="application/json"`. Using `text/csv` will fail,
because the deployed `inference.py` for this endpoint expects JSON input.

### 5.1 The serving contract, and self-configuring the UI

The cell below pulls the contract from `pipeline_lib` when `src/` is on the path — the same single
source of truth the endpoint imports — and mirrors it otherwise so this notebook still runs
standalone.

It also tries to read `preprocessing_artifacts.json`, the fitted encoder state that Notebook 03
Step 4 asserts the deployed pipeline against. When it is readable, the Gradio dropdowns, amenity
checkboxes and slider defaults are built from the **actual frozen state of the model** rather than
from assumptions. When it is not, the notebook falls back to sensible defaults and says so.

In [5]:
import os
import re
import sys
import math
import time
from datetime import date, datetime
from pathlib import Path

# --- The contract: pipeline_lib is authoritative ---------------------------------
for _p in ("src", ".", "../src"):
    if os.path.isdir(_p) and os.path.abspath(_p) not in sys.path:
        sys.path.insert(0, os.path.abspath(_p))

try:
    from pipeline_lib import (CITY_CENTERS, CURRENCY, RESPONSE_TIME_LEVELS, REVIEW_COLS,
                              SERVING_COLUMNS, SNAPSHOT_DATE, property_group)
    from pipeline_lib import __version__ as PIPELINE_LIB_VERSION
    SNAPSHOT_DATE = date(SNAPSHOT_DATE.year, SNAPSHOT_DATE.month, SNAPSHOT_DATE.day)
    CONTRACT_SOURCE = "pipeline_lib (imported from src/)"
except Exception:
    PIPELINE_LIB_VERSION = "1.0.0 (mirrored)"
    CONTRACT_SOURCE = "mirrored constants in this notebook"
    SNAPSHOT_DATE = date(2021, 3, 1)

    CITY_CENTERS = {
        "Paris": (48.8530, 2.3499), "New York": (40.7580, -73.9855),
        "Sydney": (-33.8568, 151.2153), "Rome": (41.8986, 12.4769),
        "Rio de Janeiro": (-22.9711, -43.1822), "Istanbul": (41.0054, 28.9768),
        "Mexico City": (19.4326, -99.1332), "Bangkok": (13.7460, 100.5340),
        "Cape Town": (-33.9221, 18.4231), "Hong Kong": (22.2819, 114.1582),
    }
    CURRENCY = {"Paris": "EUR", "Rome": "EUR", "New York": "USD", "Sydney": "AUD",
                "Rio de Janeiro": "BRL", "Istanbul": "TRY", "Mexico City": "MXN",
                "Bangkok": "THB", "Cape Town": "ZAR", "Hong Kong": "HKD"}
    RESPONSE_TIME_LEVELS = ["within an hour", "within a few hours", "within a day",
                            "a few days or more", "no history"]
    REVIEW_COLS = ["review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
                   "review_scores_checkin", "review_scores_communication",
                   "review_scores_location", "review_scores_value"]
    SERVING_COLUMNS = [
        "city", "latitude", "longitude", "neighbourhood", "property_type", "room_type",
        "accommodates", "bedrooms", "amenities", "minimum_nights", "maximum_nights",
        "instant_bookable", "host_since", "host_is_superhost", "host_has_profile_pic",
        "host_identity_verified", "host_total_listings_count", "host_response_time",
        "host_response_rate", "host_acceptance_rate",
    ] + REVIEW_COLS

    def property_group(pt):
        # Mirror of pipeline_lib.property_group -- display only.
        t = str(pt).lower()
        if any(k in t for k in ("hotel", "hostel", "resort")):                       return "hotel_hostel"
        if any(k in t for k in ("bed and breakfast", "guesthouse", "guest suite")):  return "bnb_guesthouse"
        if any(k in t for k in ("boat", "camper", "tent", "castle", "treehouse", "yurt",
                                "tiny house", "island", "cave", "dome", "windmill", "lighthouse",
                                "barn", "farm stay", "earth house", "hut", "tipi", "igloo",
                                "ryokan", "riad", "casa particular", "minsu", "pension",
                                "cycladic", "dammuso", "trullo", "shepherd")):       return "unique_stay"
        if any(k in t for k in ("apartment", "condominium", "loft", "serviced")):    return "apartment_condo"
        if any(k in t for k in ("house", "townhouse", "villa", "cottage", "bungalow",
                                "cabin", "chalet")):                                return "house_villa"
        if "private room" in t or "shared room" in t or "room in" in t:              return "room_other_dwelling"
        return "other"

CITIES = sorted(CITY_CENTERS)
R_EARTH_KM = 6371.0
MIN_NIGHTS_CAP, MAX_NIGHTS_CAP, BEDROOMS_CAP = 365, 1125, 16

# Fields that may legitimately be sent as null (see the table above)
NULLABLE = {"bedrooms", "host_since", "host_response_rate", "host_acceptance_rate",
            "host_response_time", "neighbourhood", *REVIEW_COLS}

# --- Fallbacks, used only when preprocessing_artifacts.json cannot be read --------
FALLBACK_ROOM_TYPES = ["Entire place", "Private room", "Shared room", "Hotel room"]
FALLBACK_PROPERTY_TYPES = [
    "Entire apartment", "Entire condominium", "Entire house", "Entire loft",
    "Entire serviced apartment", "Entire townhouse", "Entire villa",
    "Private room in apartment", "Private room in house", "Private room in condominium",
    "Private room in bed and breakfast", "Private room in guesthouse",
    "Private room in hostel", "Room in boutique hotel", "Room in hotel",
    "Shared room in apartment", "Shared room in hostel", "Boat", "Tiny house",
]
FALLBACK_AMENITIES = [
    "wifi", "kitchen", "air conditioning", "heating", "washer", "dryer", "tv",
    "essentials", "hangers", "iron", "shampoo", "hair dryer", "hot water",
    "refrigerator", "microwave", "coffee maker", "cooking basics", "oven", "stove",
    "dishes and silverware", "elevator", "free parking on premises", "pool", "gym",
    "smoke alarm", "carbon monoxide alarm", "fire extinguisher", "first aid kit",
    "long term stays allowed", "dedicated workspace",
]
FALLBACK_REVIEW_DEFAULTS = {
    "review_scores_rating": 95.0, "review_scores_accuracy": 10.0,
    "review_scores_cleanliness": 9.0, "review_scores_checkin": 10.0,
    "review_scores_communication": 10.0, "review_scores_location": 10.0,
    "review_scores_value": 9.0,
}
# Per-city test error from Notebook 03's evaluation_report.json, used to put an honest
# band around the point estimate. Replaced by the real report when it is readable.
FALLBACK_PER_CITY = {
    "Bangkok":        {"mape_pct": 37.4, "mae_ccy": 997.0, "n": 3863},
    "Cape Town":      {"mape_pct": 34.2, "mae_ccy": 957.3, "n": 3815},
    "Hong Kong":      {"mape_pct": 34.4, "mae_ccy": 324.2, "n": 1415},
    "Istanbul":       {"mape_pct": 39.0, "mae_ccy": 305.8, "n": 4893},
    "Mexico City":    {"mape_pct": 33.5, "mae_ccy": 427.3, "n": 4006},
    "New York":       {"mape_pct": 29.8, "mae_ccy":  51.9, "n": 7377},
    "Paris":          {"mape_pct": 27.6, "mae_ccy":  34.7, "n": 12916},
    "Rio de Janeiro": {"mape_pct": 44.6, "mae_ccy": 406.8, "n": 5321},
    "Rome":           {"mape_pct": 33.6, "mae_ccy":  47.6, "n": 5519},
    "Sydney":         {"mape_pct": 32.6, "mae_ccy":  88.4, "n": 6724},
}

# --- Read the published fitted state, if we can ----------------------------------
ARTIFACTS, ARTIFACTS_SOURCE, EVAL_REPORT = {}, "none - using built-in fallbacks", {}


def _load_json_local_or_s3(filename, prefixes):
    # Local folders first, then S3. Never raises.
    for folder in ("artifacts", "processed", "nb01_reference", ".", "../artifacts", "../processed"):
        p = Path(folder, filename)
        if p.is_file():
            try:
                return json.loads(p.read_text()), str(p)
            except Exception:
                pass
    for prefix in prefixes:
        key = f"{prefix}/{filename}"
        try:
            return json.loads(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()), f"s3://{BUCKET}/{key}"
        except Exception:
            continue
    return None, None


_arts, _src = _load_json_local_or_s3("preprocessing_artifacts.json",
                                     [PROCESSED_PREFIX, ARTIFACTS_PREFIX])
if _arts:
    ARTIFACTS, ARTIFACTS_SOURCE = _arts, _src

_report, _ = _load_json_local_or_s3("evaluation_report.json", [ARTIFACTS_PREFIX])
if _report:
    EVAL_REPORT = _report


def room_type_choices():
    return ARTIFACTS.get("onehot_levels", {}).get("room_type") or FALLBACK_ROOM_TYPES


def response_time_choices():
    return ARTIFACTS.get("onehot_levels", {}).get("host_response_time") or list(RESPONSE_TIME_LEVELS)


def amenity_choices():
    # The top-30 vocabulary the encoder actually one-hots.
    return ARTIFACTS.get("amenity_vocab") or FALLBACK_AMENITIES


def known_neighbourhoods(city):
    freq = ARTIFACTS.get("neighbourhood_freq") or {}
    return sorted({k.split("||", 1)[1] for k in freq if k.split("||", 1)[0] == city})


def neighbourhood_is_known(city, neighbourhood):
    # True / False, or None when we have no frequency table to check against.
    freq = ARTIFACTS.get("neighbourhood_freq")
    if not freq:
        return None
    return f"{city}||{neighbourhood or 'unknown'}" in freq


def review_defaults():
    # Slider defaults = the encoder's fitted imputation medians, when available.
    med = ARTIFACTS.get("imputation_values") or {}
    return {c: float(med.get(c, FALLBACK_REVIEW_DEFAULTS[c])) for c in REVIEW_COLS}


def review_slider_max(col, default_value):
    # review_scores_rating is 0-100 in this dataset; the six sub-scores are 0-10.
    # Infer from the fitted median so the UI follows the data, not an assumption.
    if col != "review_scores_rating":
        return 10.0
    return 100.0 if default_value > 10 else (10.0 if default_value > 5 else 5.0)


def per_city_error(city):
    for row in EVAL_REPORT.get("per_city", []) or []:
        if row.get("city") == city:
            return row
    return FALLBACK_PER_CITY.get(city)


print(f"Contract     : {len(SERVING_COLUMNS)} raw fields, from {CONTRACT_SOURCE}")
print(f"               pipeline_lib v{PIPELINE_LIB_VERSION}")
print(f"Fitted state : {ARTIFACTS_SOURCE}")
if ARTIFACTS:
    print(f"               {len(amenity_choices())} amenities in vocabulary")
    print(f"               {len(ARTIFACTS.get('neighbourhood_freq') or {})} (city, neighbourhood) pairs")
    print(f"               {len(ARTIFACTS.get('feature_columns') or [])} feature columns")
    print(f"               room types: {room_type_choices()}")
else:
    print("               Dropdowns will use built-in defaults, which may not match the")
    print("               frozen encoder levels. Run Notebook 03 first, or from the same")
    print("               directory, so artifacts/preprocessing_artifacts.json is readable.")

Contract     : 27 raw fields, from pipeline_lib (imported from src/)
               pipeline_lib v1.0.0
Fitted state : processed/preprocessing_artifacts.json
               30 amenities in vocabulary
               651 (city, neighbourhood) pairs
               88 feature columns
               room types: ['Entire place', 'Hotel room', 'Private room', 'Shared room']


### 5.2 Two example listings, sent straight to the endpoint

A Paris private room and a Bangkok condo — the same two markets Notebook 03 Step 8.2 used for its
local smoke test. The first call after an idle period is slow: the serverless endpoint scales to
zero, so it has to pull the container and deserialize the pipeline in `model_fn`. That shows up in
the `ModelSetupTime` CloudWatch metric and can take up to about a minute.

In [6]:
paris_listing = {
    "city": "Paris",
    "latitude": 48.8867,
    "longitude": 2.3431,
    "neighbourhood": "Buttes-Montmartre",
    "property_type": "Private room in apartment",
    "room_type": "Private room",
    "accommodates": 2,
    "bedrooms": 1,
    "amenities": json.dumps(["essentials", "hangers", "heating", "hot water", "iron",
                             "kitchen", "long term stays allowed", "shampoo", "washer", "wifi"]),
    "minimum_nights": 2,
    "maximum_nights": 90,
    "instant_bookable": "f",
    "host_since": "2015-06-12",
    "host_is_superhost": "f",
    "host_has_profile_pic": "t",
    "host_identity_verified": "t",
    "host_total_listings_count": 1,
    "host_response_time": "within a few hours",
    "host_response_rate": 90,
    "host_acceptance_rate": 80,
    "review_scores_rating": 95,
    "review_scores_accuracy": 10,
    "review_scores_cleanliness": 9,
    "review_scores_checkin": 10,
    "review_scores_communication": 10,
    "review_scores_location": 10,
    "review_scores_value": 9,
}

bangkok_listing = dict(paris_listing, **{
    "city": "Bangkok",
    "latitude": 13.7230,
    "longitude": 100.5680,
    "neighbourhood": "Khlong Toei",
    "property_type": "Entire condominium",
    "room_type": "Entire place",
    "accommodates": 4,
    "bedrooms": 2,
    "amenities": json.dumps(["air conditioning", "dryer", "elevator", "essentials", "gym",
                             "kitchen", "pool", "tv", "washer", "wifi"]),
    "minimum_nights": 1,
    "maximum_nights": 365,
    "instant_bookable": "t",
    "host_since": "2017-03-08",
    "host_is_superhost": "t",
    "host_total_listings_count": 6,
    "host_response_time": "within an hour",
    "host_response_rate": 100,
    "host_acceptance_rate": 95,
})


def invoke_airbnb_endpoint(payload, endpoint_name=ENDPOINT_NAME):
    # POST the raw listing contract. Returns (records, latency_ms).
    t0 = time.perf_counter()
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload)
    )
    records = json.loads(response["Body"].read())
    return records, (time.perf_counter() - t0) * 1000.0


# Contract check before we spend a round trip on it
for name, listing in [("paris_listing", paris_listing), ("bangkok_listing", bangkok_listing)]:
    missing = [c for c in SERVING_COLUMNS if c not in listing]
    assert not missing, f"{name} is missing required fields: {missing}"
print(f"Both example listings carry all {len(SERVING_COLUMNS)} contract fields.\n")

try:
    # One request, both listings -- input_fn accepts a list as well as a single object
    records, ms = invoke_airbnb_endpoint([paris_listing, bangkok_listing])

    for rec in records:
        print(f"{rec['city']:<10s} suggested fair rate {rec['suggested_nightly_price']:>10,.2f} "
              f"{rec['currency']}   (log_price {rec['log_price']})")
    print(f"\nRound trip: {ms:,.0f} ms")

except Exception as e:
    print("Endpoint invocation failed.")
    print("Error:", e)

Both example listings carry all 27 contract fields.



Paris      suggested fair rate      47.86 EUR   (log_price 3.889)
Bangkok    suggested fair rate   1,671.61 THB   (log_price 7.4221)

Round trip: 7,337 ms


## 6. Build the Gradio Prediction Function

This function will be connected to the Gradio interface. It collects values from UI fields, maps
them onto the 27-field contract, sends them to the endpoint, and returns the endpoint response.

The UI has more controls than the contract has fields — 33 versus 27 — because some contract fields
need two controls. "Bedrooms not stated", "Host since: not known" and "This listing has reviews"
are toggles that decide whether a field is sent as a **value or as `null`**, which is how a student
exercises the missing-value flags inside the pipeline.

Rather than a 33-parameter function signature, the handler takes `*values` and zips them against
`UI_FIELDS`. The order of `UI_FIELDS` is the order of the `inputs=` list in Section 7, and it is the
only place the two have to agree.

In [7]:
import boto3
import json

REGION = "ap-southeast-1"

# change to your own team including ENDPOINT
TEAM_ID       = "team14"
STUDENT_ID    = "s1402"
ENDPOINT_NAME = "iti113-team14-airbnb-price-slsv2"

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

### 6.1 UI state → the serving contract

`build_payload` is the only place the contract is written down. The UI can be rearranged freely
without touching it, and `validate` catches the inputs the pipeline cannot recover from *before*
spending a round trip on a `ModelError`.

In [8]:
# The order here is the order of the inputs= list in Section 7.
UI_FIELDS = [
    "city", "neighbourhood", "latitude", "longitude",
    "property_type", "room_type", "accommodates", "bedrooms", "bedrooms_unknown",
    "amenities_selected", "amenities_extra",
    "minimum_nights", "maximum_nights", "instant_bookable",
    "host_since", "host_since_unknown", "host_is_superhost", "host_has_profile_pic",
    "host_identity_verified", "host_total_listings_count", "host_response_time",
    "host_response_rate", "host_response_rate_known",
    "host_acceptance_rate", "host_acceptance_rate_known",
    "has_reviews", *REVIEW_COLS,
]


def _tf(flag):
    # The raw platform encoding for booleans: 't' / 'f', exactly as trained.
    return "t" if bool(flag) else "f"


def _amenities(selected, extra):
    items = list(selected or [])
    for token in re.split(r"[,;\n]", extra or ""):
        if token.strip():
            items.append(token.strip())
    return sorted({a.strip().lower() for a in items if a.strip()})


def build_payload(ui):
    # UI state -> one JSON object on the 27-field SERVING_COLUMNS contract.
    has_reviews = bool(ui.get("has_reviews", True))

    payload = {
        "city": ui["city"],
        "latitude": float(ui["latitude"]),
        "longitude": float(ui["longitude"]),
        "neighbourhood": (ui.get("neighbourhood") or "").strip() or None,
        "property_type": (ui.get("property_type") or "").strip() or "Other",
        "room_type": ui["room_type"],
        "accommodates": int(ui["accommodates"]),
        "bedrooms": None if ui.get("bedrooms_unknown") else int(ui["bedrooms"]),
        "amenities": json.dumps(_amenities(ui.get("amenities_selected"), ui.get("amenities_extra"))),
        "minimum_nights": int(ui["minimum_nights"]),
        "maximum_nights": int(ui["maximum_nights"]),
        "instant_bookable": _tf(ui.get("instant_bookable")),
        "host_since": None if ui.get("host_since_unknown") else (ui.get("host_since") or "").strip() or None,
        "host_is_superhost": _tf(ui.get("host_is_superhost")),
        "host_has_profile_pic": _tf(ui.get("host_has_profile_pic")),
        "host_identity_verified": _tf(ui.get("host_identity_verified")),
        "host_total_listings_count": int(ui.get("host_total_listings_count") or 0),
        "host_response_time": ui.get("host_response_time") or None,
        "host_response_rate": float(ui["host_response_rate"]) if ui.get("host_response_rate_known") else None,
        "host_acceptance_rate": float(ui["host_acceptance_rate"]) if ui.get("host_acceptance_rate_known") else None,
    }
    for col in REVIEW_COLS:
        payload[col] = float(ui[col]) if has_reviews else None

    # Order the keys as the contract declares them: cosmetic for the JSON view,
    # and it makes a missing field obvious at a glance.
    return {k: payload[k] for k in SERVING_COLUMNS}


def validate(payload):
    # Fail fast, in the UI, on the inputs the pipeline cannot recover from.
    problems = []
    if payload["city"] not in CITY_CENTERS:
        problems.append("City must be one of the 10 markets the model was trained on: "
                        + ", ".join(CITIES) + ".")
    for field in ("latitude", "longitude", "accommodates", "minimum_nights", "maximum_nights"):
        v = payload.get(field)
        if v is None or (isinstance(v, float) and math.isnan(v)):
            problems.append(f"{field} is required - the pipeline does not impute it.")

    # 0 is a value, not an absence: test against None, never truthiness.
    if payload["accommodates"] is not None and payload["accommodates"] < 1:
        problems.append("Accommodates must be at least 1; a listing for 0 guests is not rentable.")
    if payload["minimum_nights"] is not None and payload["minimum_nights"] < 1:
        problems.append("Minimum nights must be at least 1.")
    if (payload["minimum_nights"] is not None and payload["maximum_nights"] is not None
            and payload["minimum_nights"] > payload["maximum_nights"]):
        problems.append("Minimum nights is greater than maximum nights.")
    if payload["bedrooms"] is not None and payload["bedrooms"] < 0:
        problems.append("Bedrooms cannot be negative. Tick 'Bedrooms not stated' to send null.")

    for key in SERVING_COLUMNS:
        if key not in payload:
            problems.append(f"Missing contract field: {key}.")
        elif payload[key] is None and key not in NULLABLE:
            problems.append(f"{key} cannot be null on this contract.")
    return problems

### 6.2 Showing the invisible half

The 27 fields a student types are not the 88 columns the model scores. `derived_preview` mirrors
`pipeline_lib`'s stateless engineering step so the UI can display the features that never appear in
the payload — including whether the `(city, neighbourhood)` pair is one the encoder has ever seen,
which is exactly the unseen-neighbourhood signal the weekly drift job counts.

This mirror is **display only**. The endpoint recomputes all of it authoritatively, and the preview
is wrapped so a bug in it can never take down a real prediction.

In [9]:
def haversine_km(lat1, lon1, lat2, lon2):
    p = math.pi / 180.0
    a = (math.sin((lat2 - lat1) * p / 2) ** 2
         + math.cos(lat1 * p) * math.cos(lat2 * p) * math.sin((lon2 - lon1) * p / 2) ** 2)
    return 2 * R_EARTH_KM * math.asin(math.sqrt(a))


def derived_preview(payload):
    # Display only. Wrapped so it can never break a response.
    try:
        return _derived_preview(payload)
    except Exception as e:
        return f"Preview unavailable ({e.__class__.__name__}). The prediction above is unaffected."


def _derived_preview(payload):
    city = payload["city"]
    lines = []

    if city in CITY_CENTERS:
        clat, clon = CITY_CENTERS[city]
        dist = haversine_km(payload["latitude"], payload["longitude"], clat, clon)
        lines.append(f"| `dist_center_km` | {dist:,.3f} km from the {city} centre |")
        lines.append(f"| `lat_coarse` / `lon_coarse` | {round(payload['latitude'], 2)}, "
                     f"{round(payload['longitude'], 2)} - exact GPS is destroyed inside the "
                     f"pipeline and never reaches an artifact |")

    amen = json.loads(payload["amenities"])
    in_vocab = len([a for a in amen if a in set(amenity_choices())])
    lines.append(f"| `amenity_count` | {len(amen)} |")
    lines.append(f"| `rare_amenity_count` | {len(amen) - in_vocab} outside the top-30 vocabulary |")

    known = neighbourhood_is_known(city, payload["neighbourhood"])
    if known is True:
        lines.append(f"| `neighbourhood_freq` | ({city}, {payload['neighbourhood']}) is in the "
                     f"fitted frequency table |")
    elif known is False:
        lines.append(f"| `neighbourhood_freq` | **0.0 - unseen (city, neighbourhood) pair.** "
                     f"This is exactly what the weekly drift job counts |")
    else:
        lines.append("| `neighbourhood_freq` | unknown here - publish "
                     "`preprocessing_artifacts.json` to check the pair |")

    if payload["host_since"]:
        try:
            hs = datetime.strptime(payload["host_since"], "%Y-%m-%d").date()
            tenure = (SNAPSHOT_DATE - hs).days
            note = "" if tenure >= 0 else " **negative - after the snapshot date, out of distribution**"
            lines.append(f"| `host_tenure_days` | {tenure:,} days as of the "
                         f"{SNAPSHOT_DATE:%Y-%m-%d} snapshot{note} |")
        except ValueError:
            lines.append("| `host_tenure_days` | host_since is not YYYY-MM-DD; it will parse to "
                         "NaT and be imputed |")
    else:
        lines.append("| `host_tenure_days` | imputed to the fitted median (host_since is null) |")

    if payload["bedrooms"] is None:
        lines.append("| `bedrooms_missing` | 1 - bedrooms filled as ceil(accommodates / 2) |")
    else:
        ppb = payload["accommodates"] / max(payload["bedrooms"], 1)
        lines.append(f"| `persons_per_bedroom` | {ppb:.3f} |")

    lines.append(f"| `is_monthly_min` | {int(payload['minimum_nights'] >= 28)} - "
                 f"minimum_nights is {payload['minimum_nights']}, the threshold is 28 |")
    lines.append(f"| `property_group` | `{property_group(payload['property_type'])}` "
                 f"- one of 7 governed buckets |")
    lines.append(f"| `has_review_scores` | {int(payload['review_scores_rating'] is not None)} |")
    missing_host = [f"`{k}`" for k in ("host_response_rate", "host_acceptance_rate")
                    if payload[k] is None]
    if missing_host:
        lines.append(f"| missing-value flags | {', '.join(missing_host)} null -> flag set, "
                     f"value imputed |")

    return ("Derived inside the endpoint, from the fields above:\n\n"
            "| feature | value |\n|---|---|\n" + "\n".join(lines))


def explain_error(exc):
    # Say what went wrong and what to do about it.
    name, msg = exc.__class__.__name__, str(exc)
    hint = ""
    if "ValidationException" in name + msg and "endpoint" in msg.lower():
        hint = (f"No endpoint named `{ENDPOINT_NAME}` in `{REGION}`. Re-run Section 2 and set "
                f"ENDPOINT_NAME to a name from that list.")
    elif "AccessDenied" in name + msg:
        hint = ("Your execution role cannot invoke this endpoint. Check that you are in your own "
                "Studio profile and that the endpoint belongs to your team.")
    elif "ModelError" in name + msg:
        hint = ("The endpoint received the request but `inference.py` rejected it - usually a "
                "contract mismatch. Compare the request JSON below against the 27 "
                "`SERVING_COLUMNS` fields.")
    elif "Throttling" in name + msg or "TooManyRequests" in msg:
        hint = "The serverless endpoint is at `max_concurrency=5`. Wait a moment and send again."
    elif "ExpiredToken" in msg or "InvalidClientTokenId" in msg or "NoCredentials" in name:
        hint = "AWS credentials are missing or expired. Restart the kernel or refresh your session."
    elif "ReadTimeout" in name or "timed out" in msg.lower():
        hint = ("This looks like a serverless cold start: the container pulls the image and "
                "deserializes the pipeline in `model_fn`, which can take up to ~60 s on the first "
                "call. Send the request again.")
    return f"### Request failed\n\n`{name}`\n\n```\n{msg}\n```\n\n{hint}".rstrip()

### 6.3 The Gradio handler

Returns four things: the result card, the derived-features table, the exact JSON that was posted,
and the raw endpoint response. Showing the request back to the student is the point — it is the
contract, and being able to read it is what makes a `ModelError` diagnosable.

The result carries a **±band from the per-city test MAPE**, not just a point estimate. A regression
model that answers "70.90 EUR" to three significant figures invites more confidence than 0.87 R²
on log-price deserves; Paris is ±27.6% and Rio is ±44.6%, and the UI says so.

In [10]:
def predict_airbnb_price(*values):
    # Gradio click handler. Returns (result card, derived table, request JSON, response JSON).
    ui = dict(zip(UI_FIELDS, values))

    try:
        payload = build_payload(ui)
    except (TypeError, ValueError) as e:
        return (f"### Cannot build the request\n\n{e}\n\nCheck the numeric fields.", "", "", "")

    request_json = json.dumps(payload, indent=2)

    problems = validate(payload)
    if problems:
        bullets = "\n".join(f"- {p}" for p in problems)
        return (f"### Fix these before sending\n\n{bullets}", "", request_json, "")

    try:
        records, ms = invoke_airbnb_endpoint(payload)
    except Exception as e:
        return (explain_error(e), derived_preview(payload), request_json, "")

    rec = records[0]
    price, ccy = rec["suggested_nightly_price"], rec["currency"]

    band = ""
    err = per_city_error(rec["city"])
    if err:
        mape = float(err["mape_pct"])
        lo, hi = price * (1 - mape / 100), price * (1 + mape / 100)
        band = (f"<div class='band'>Typical error for {rec['city']} is "
                f"<strong>&plusmn;{mape:.1f}%</strong> on {int(err.get('n', 0)):,} held-out test "
                f"listings, so read this as roughly <strong>{lo:,.0f}&ndash;{hi:,.0f} {ccy}</strong>, "
                f"not a precise figure.</div>")

    card = (
        "<div class='result'>"
        f"<div class='eyebrow'>Suggested fair advertised nightly rate &middot; {rec['city']}</div>"
        f"<div class='price'>{price:,.2f} <span class='ccy'>{ccy}</span></div>"
        f"{band}"
        f"<div class='meta'>log_price {rec['log_price']} &middot; {ms:,.0f} ms round trip "
        f"&middot; {ENDPOINT_NAME}</div>"
        "</div>"
    )
    return card, derived_preview(payload), request_json, json.dumps(records, indent=2)


# --- Example listings for the UI buttons (see Section 8) --------------------------
def _base_preset(**overrides):
    d = review_defaults()
    preset = {
        "city": "Paris", "neighbourhood": "Buttes-Montmartre",
        "latitude": 48.8867, "longitude": 2.3431,
        "property_type": "Private room in apartment", "room_type": "Private room",
        "accommodates": 2, "bedrooms": 1, "bedrooms_unknown": False,
        "amenities_selected": [], "amenities_extra": "",
        "minimum_nights": 2, "maximum_nights": 90, "instant_bookable": False,
        "host_since": "2015-06-12", "host_since_unknown": False,
        "host_is_superhost": False, "host_has_profile_pic": True,
        "host_identity_verified": True, "host_total_listings_count": 1,
        "host_response_time": "within a few hours",
        "host_response_rate": 90.0, "host_response_rate_known": True,
        "host_acceptance_rate": 80.0, "host_acceptance_rate_known": True,
        "has_reviews": True, **{c: d[c] for c in REVIEW_COLS},
    }
    preset.update(overrides)
    return preset


def _amen(*names):
    # Split a listing's amenities across the two UI controls. Anything in the fitted
    # vocabulary becomes a ticked checkbox; the rest goes to the free-text box, where it
    # still raises amenity_count and rare_amenity_count. A real listing does not lose an
    # amenity just because the encoder has no flag for it.
    vocab = set(amenity_choices())
    return {"amenities_selected": [n for n in names if n in vocab],
            "amenities_extra": ", ".join(n for n in names if n not in vocab)}


PRESETS = {
    "Paris studio": _base_preset(
        **_amen("wifi", "heating", "kitchen", "essentials", "hangers", "iron", "shampoo",
                "washer", "hot water", "long term stays allowed"),
    ),
    "Bangkok condo": _base_preset(
        city="Bangkok", neighbourhood="Khlong Toei", latitude=13.7230, longitude=100.5680,
        property_type="Entire condominium", room_type="Entire place",
        accommodates=4, bedrooms=2,
        **_amen("wifi", "air conditioning", "pool", "gym", "kitchen", "washer", "dryer", "tv",
                "elevator", "free parking on premises", "essentials", "hot water",
                "dedicated workspace", "refrigerator", "microwave"),
        minimum_nights=1, maximum_nights=365, instant_bookable=True,
        host_since="2017-03-08", host_is_superhost=True, host_total_listings_count=6,
        host_response_time="within an hour", host_response_rate=100.0, host_acceptance_rate=95.0,
    ),
    "New York, brand new listing": _base_preset(
        city="New York", neighbourhood="Midtown", latitude=40.7549, longitude=-73.9840,
        property_type="Entire apartment", room_type="Entire place",
        accommodates=4, bedrooms=1,
        bedrooms_unknown=True,              # -> bedrooms_missing
        **_amen("wifi", "kitchen", "air conditioning", "heating", "elevator", "essentials", "tv"),
        minimum_nights=30,                  # -> is_monthly_min
        maximum_nights=365, instant_bookable=True,
        host_since="", host_since_unknown=True,   # -> host_tenure_days imputed
        host_has_profile_pic=True, host_identity_verified=False, host_total_listings_count=1,
        host_response_time="no history",
        host_response_rate_known=False,     # -> host_response_rate_missing
        host_acceptance_rate_known=False,
        has_reviews=False,                  # -> has_review_scores = 0
    ),
}

# Every preset must cover every UI control, or the loader buttons would leave stale values behind.
for _name, _p in PRESETS.items():
    assert set(_p) == set(UI_FIELDS), f"{_name}: {set(UI_FIELDS) ^ set(_p)}"

print(f"Prediction function ready. {len(UI_FIELDS)} UI controls -> "
      f"{len(SERVING_COLUMNS)} contract fields.")
print(f"Presets: {list(PRESETS)}")

Prediction function ready. 33 UI controls -> 27 contract fields.
Presets: ['Paris studio', 'Bangkok condo', 'New York, brand new listing']


## 7. Launch the Gradio UI

Run the cell below to start the front-end UI inside the notebook.

The 27 fields are grouped into five tabs — Listing, Amenities, Booking rules, Host, Reviews — because
a flat wall of 27 inputs is unusable. Picking a city snaps the coordinates to that city's centre and
reloads the neighbourhood list, so the most common way to produce a `NaN` (a coordinate that belongs
to no trained market) cannot happen by accident.

> If the UI does not appear inline, check the printed local URL. In managed Studio environments,
> external sharing may be disabled, so `share=True` may need to become `share=False`.

In [11]:
CSS = '''
.result {padding: 20px 22px; border: 1px solid var(--border-color-primary);
         border-radius: 10px; background: var(--background-fill-secondary);}
.result .eyebrow {font-size: .78rem; letter-spacing: .09em; text-transform: uppercase;
         opacity: .65; margin-bottom: 6px;}
.result .price {font-size: 2.6rem; font-weight: 650; line-height: 1.1;
         font-variant-numeric: tabular-nums;}
.result .price .ccy {font-size: 1.1rem; font-weight: 500; opacity: .7; margin-left: 4px;}
.result .band {margin-top: 12px; padding: 10px 12px; border-radius: 8px;
         background: var(--background-fill-primary); font-size: .9rem; line-height: 1.5;}
.result .meta {margin-top: 12px; font-size: .78rem; opacity: .6;
         font-variant-numeric: tabular-nums;}
'''

rdef = review_defaults()

try:
    status = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
except Exception:
    status = "NotFound"
status_note = {
    "InService": "endpoint is InService",
    "Creating": "endpoint is still provisioning - predictions will fail until it is InService",
    "NotFound": f"no endpoint named `{ENDPOINT_NAME}` in `{REGION}`",
}.get(status, f"endpoint status: {status}")

fitted_note = (f"UI configured from `{ARTIFACTS_SOURCE}`" if ARTIFACTS
               else "`preprocessing_artifacts.json` not found - dropdowns use built-in defaults, "
                    "which may not match the frozen encoder levels")

with gr.Blocks(title=f"{COURSE} {TEAM_ID} - Airbnb fair nightly rate", css=CSS) as demo:
    gr.Markdown(
        f'''
        # Airbnb fair advertised nightly rate

        **{COURSE} {TEAM_ID} / {STUDENT_ID}** &nbsp;&middot;&nbsp; region `{REGION}`
        &nbsp;&middot;&nbsp; endpoint `{ENDPOINT_NAME}` &nbsp;&middot;&nbsp; {status_note}

        This front-end trains nothing. It collects the {len(SERVING_COLUMNS)} raw listing fields of
        the `SERVING_COLUMNS` contract, posts them as JSON to the SageMaker Serverless Endpoint, and
        shows what comes back. All feature engineering happens inside the endpoint, in the same
        fitted pipeline object that training serialized.

        {fitted_note}.
        '''
    )

    c = {}

    with gr.Row():
        with gr.Column(scale=3):

            with gr.Tab("Listing"):
                with gr.Row():
                    c["city"] = gr.Dropdown(
                        label="City", choices=CITIES, value="Paris",
                        info="One of the 10 trained markets; sets the currency and the "
                             "city-centre reference point")
                    c["neighbourhood"] = gr.Dropdown(
                        label="Neighbourhood", choices=known_neighbourhoods("Paris") or [],
                        value="Buttes-Montmartre", allow_custom_value=True,
                        info="Type any value; unseen pairs get frequency 0.0")
                with gr.Row():
                    c["latitude"] = gr.Number(label="Latitude", value=48.8867)
                    c["longitude"] = gr.Number(label="Longitude", value=2.3431)
                gr.Markdown(
                    "<small>Coordinates are used to derive distance from the city centre, then "
                    "coarsened to 2 decimal places and discarded. Exact GPS never reaches a "
                    "stored artifact.</small>")
                with gr.Row():
                    c["property_type"] = gr.Dropdown(
                        label="Property type", choices=FALLBACK_PROPERTY_TYPES,
                        value="Private room in apartment", allow_custom_value=True,
                        info="Free text; mapped to one of 7 governed groups")
                    c["room_type"] = gr.Dropdown(
                        label="Room type", choices=room_type_choices(), value="Private room")
                with gr.Row():
                    c["accommodates"] = gr.Number(label="Accommodates", value=2, precision=0, minimum=1)
                    c["bedrooms"] = gr.Number(label="Bedrooms", value=1, precision=0,
                                              minimum=0, maximum=BEDROOMS_CAP)
                    c["bedrooms_unknown"] = gr.Checkbox(label="Bedrooms not stated", value=False)

            with gr.Tab("Amenities"):
                c["amenities_selected"] = gr.CheckboxGroup(
                    label="Amenities in the model's vocabulary",
                    choices=amenity_choices(), value=[],
                    info="Each of these is a one-hot flag in the 88-column matrix")
                c["amenities_extra"] = gr.Textbox(
                    label="Other amenities", lines=2, placeholder="balcony, sea view, piano",
                    info="Comma separated. These do not get their own flag, but they raise "
                         "amenity_count and rare_amenity_count.")

            with gr.Tab("Booking rules"):
                with gr.Row():
                    c["minimum_nights"] = gr.Number(
                        label="Minimum nights", value=2, precision=0, minimum=1,
                        info=f"Winsorised at {MIN_NIGHTS_CAP}; >= 28 sets is_monthly_min")
                    c["maximum_nights"] = gr.Number(
                        label="Maximum nights", value=90, precision=0, minimum=1,
                        info=f"Winsorised at {MAX_NIGHTS_CAP}")
                c["instant_bookable"] = gr.Checkbox(label="Instant bookable", value=False)

            with gr.Tab("Host"):
                with gr.Row():
                    c["host_since"] = gr.Textbox(
                        label="Host since", value="2015-06-12", placeholder="YYYY-MM-DD",
                        info=f"Tenure is measured against the {SNAPSHOT_DATE:%Y-%m-%d} data snapshot")
                    c["host_since_unknown"] = gr.Checkbox(label="Not known", value=False)
                with gr.Row():
                    c["host_is_superhost"] = gr.Checkbox(label="Superhost", value=False)
                    c["host_has_profile_pic"] = gr.Checkbox(label="Has profile picture", value=True)
                    c["host_identity_verified"] = gr.Checkbox(label="Identity verified", value=True)
                with gr.Row():
                    c["host_total_listings_count"] = gr.Number(
                        label="Total listings by this host", value=1, precision=0, minimum=0)
                    c["host_response_time"] = gr.Dropdown(
                        label="Response time", choices=response_time_choices(),
                        value="within a few hours")
                with gr.Row():
                    c["host_response_rate"] = gr.Slider(
                        label="Response rate (%)", minimum=0, maximum=100, step=1, value=90)
                    c["host_response_rate_known"] = gr.Checkbox(label="Known", value=True)
                with gr.Row():
                    c["host_acceptance_rate"] = gr.Slider(
                        label="Acceptance rate (%)", minimum=0, maximum=100, step=1, value=80)
                    c["host_acceptance_rate_known"] = gr.Checkbox(label="Known", value=True)

            with gr.Tab("Reviews"):
                c["has_reviews"] = gr.Checkbox(
                    label="This listing has reviews", value=True,
                    info="Unchecked sends nulls for all seven scores, which sets "
                         "has_review_scores = 0 and imputes the fitted medians")
                for col in REVIEW_COLS:
                    c[col] = gr.Slider(
                        label=col.replace("review_scores_", "").replace("_", " ").title(),
                        minimum=0, maximum=review_slider_max(col, rdef[col]),
                        step=1 if col == "review_scores_rating" else 0.1,
                        value=rdef[col])

        with gr.Column(scale=2):
            gr.Markdown("### Example listings")
            preset_buttons = {name: gr.Button(name, size="sm") for name in PRESETS}
            predict_button = gr.Button("Get suggested rate", variant="primary", size="lg")
            result = gr.HTML(
                "<div class='result'><div class='eyebrow'>No request sent yet</div>"
                "<div class='meta'>Fill in the listing, or load an example, then get a suggested "
                "rate. The first call after an idle period takes longer: the serverless endpoint "
                "scales to zero, so it has to pull the container and deserialize the pipeline."
                "</div></div>")
            with gr.Accordion("What the endpoint derives from this request", open=False):
                derived = gr.Markdown()
            with gr.Accordion("Request JSON (the 27-field contract)", open=False):
                request_view = gr.Code(language="json", label="POST body")
            with gr.Accordion("Endpoint response", open=False):
                response_view = gr.Code(language="json", label="Response")

    inputs = [c[k] for k in UI_FIELDS]
    outputs = [result, derived, request_view, response_view]

    predict_button.click(fn=predict_airbnb_price, inputs=inputs, outputs=outputs)

    # City drives the currency, the centre point and the neighbourhood list.
    def on_city(city):
        lat, lon = CITY_CENTERS.get(city, (0.0, 0.0))
        choices = known_neighbourhoods(city)
        return (gr.update(choices=choices, value=(choices[0] if choices else None)),
                gr.update(value=lat),
                gr.update(value=lon))

    c["city"].change(fn=on_city, inputs=c["city"],
                     outputs=[c["neighbourhood"], c["latitude"], c["longitude"]])

    def make_loader(name):
        def _load():
            p = PRESETS[name]
            return [gr.update(value=p[k]) for k in UI_FIELDS]
        return _load

    for _name, _button in preset_buttons.items():
        _button.click(fn=make_loader(_name), inputs=None, outputs=inputs)

    gr.Markdown(
        f'''
        ---
        **Contract** &middot; `{len(SERVING_COLUMNS)}` raw fields in, one record out:
        `suggested_nightly_price`, `currency`, `city`, `log_price`. Nullable fields: `bedrooms`,
        `host_since`, `host_response_rate`, `host_acceptance_rate`, `host_response_time`,
        `neighbourhood` and the seven review scores - each has an explicit missing-value flag or
        imputation inside the pipeline. Everything else must carry a value.

        **Cost** &middot; serverless, 3072 MB, `max_concurrency=5`. Roughly $0 when idle, billed per
        invocation. Cold starts show up in the round-trip figure above and in the `ModelSetupTime`
        CloudWatch metric.
        '''
    )

# In SageMaker Studio, inline=True is usually the most convenient for notebook demos.

# if below fails to show:
# demo.launch(
#     inline=True,
#     share=False,
#     debug=True,
#     server_name="0.0.0.0",
#     server_port=7860
# )
# run this temporarily
demo.launch(
    inline=True,
    share=True,
    debug=True
)

/tmp/ipykernel_1093/2751403697.py:31: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title=f"{COURSE} {TEAM_ID} - Airbnb fair nightly rate", css=CSS) as demo:


* Running on local URL:  http://127.0.0.1:7860


* Running on public URL: https://704d537ed2b6a40726.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.close()  # to close gradio
# gradio does not close, press stop in jupyter notebook

In [ ]:
# run this to confirm gradio closed
import os
import signal
import subprocess

PORT = 7860

cmd = f"lsof -ti:{PORT}"
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

pids = result.stdout.strip().splitlines()

if not pids:
    print(f"No process found on port {PORT}")
else:
    for pid in pids:
        print(f"Killing process {pid} on port {PORT}")
        os.kill(int(pid), signal.SIGTERM)

    print(f"Port {PORT} should now be free.")

## 8. Example Inputs for Students

The three buttons in the UI load these. Between them they exercise every branch of the contract.

### Paris studio — a fully populated listing

| Feature | Value |
|---|---:|
| city | Paris |
| neighbourhood | Buttes-Montmartre |
| latitude / longitude | 48.8867 / 2.3431 |
| property_type | Private room in apartment |
| room_type | Private room |
| accommodates | 2 |
| bedrooms | 1 |
| minimum_nights / maximum_nights | 2 / 90 |
| instant_bookable | f |
| host_since | 2015-06-12 |
| host_is_superhost | f |
| host_total_listings_count | 1 |
| host_response_time | within a few hours |
| host_response_rate / host_acceptance_rate | 90 / 80 |
| review_scores_rating | 95 |

### Bangkok condo — entire place, high amenity count, superhost

| Feature | Value |
|---|---:|
| city | Bangkok |
| neighbourhood | Khlong Toei |
| latitude / longitude | 13.7230 / 100.5680 |
| property_type | Entire condominium |
| room_type | Entire place |
| accommodates | 4 |
| bedrooms | 2 |
| minimum_nights / maximum_nights | 1 / 365 |
| instant_bookable | t |
| host_since | 2017-03-08 |
| host_is_superhost | t |
| host_total_listings_count | 6 |
| host_response_time | within an hour |
| host_response_rate / host_acceptance_rate | 100 / 95 |
| review_scores_rating | 95 |

### New York, brand new listing — every missing-value flag fires

This is the interesting one. It sends `null` for `bedrooms`, `host_since`, both host rates and all
seven review scores, and sets `minimum_nights = 30`.

| Feature | Value | What it exercises |
|---|---:|---|
| bedrooms | `null` | `bedrooms_missing = 1`, filled as `ceil(accommodates / 2)` |
| host_since | `null` | `host_tenure_days` imputed to the fitted median |
| host_response_rate | `null` | `host_response_rate_missing = 1` |
| host_acceptance_rate | `null` | `host_acceptance_rate_missing = 1` |
| host_response_time | no history | the fifth one-hot level |
| review_scores_* | `null` × 7 | `has_review_scores = 0` |
| minimum_nights | 30 | `is_monthly_min = 1` |

A brand new listing with no reviews and no host history is not an edge case to be defended against —
it is the single most valuable request this model can serve, because a host with no reviews has no
comparable listings of their own to price against. The pipeline was built to answer it.

### Things worth trying

- Move the coordinates far from the city centre and watch `dist_center_km` change the answer.
- Type a neighbourhood that does not exist. The prediction still works; the derived panel flags the
  pair as unseen, and `neighbourhood_freq` falls to 0.0. That is the drift signal, visible.
- Set `host_since` to a date after **2021-03-01**. Tenure goes negative, because the training data is
  a snapshot as of that date. The endpoint will answer anyway — models do not refuse
  out-of-distribution input, which is exactly why monitoring exists.

## 9. Troubleshooting

| Problem | Possible Cause | What to Check |
|---|---|---|
| `ValidationException: Could not find endpoint` | Wrong endpoint name or region | Re-run Section 2 and copy the exact endpoint name. |
| `AccessDeniedException` | Wrong team role, or endpoint outside team permission | Check that you are using your own SageMaker Studio profile and your own team endpoint. |
| `ModelError` | Payload does not match `inference.py` | Open the "Request JSON" panel and compare it against the 27 `SERVING_COLUMNS` fields. |
| `ModelError` mentioning NaNs | A non-nullable field was sent as `null` | See the nullability table in Section 5. Only 12 of the 27 fields accept `null`. |
| First prediction takes 30–60 s | Serverless cold start | Expected. The endpoint scales to zero; watch `ModelSetupTime` in CloudWatch. |
| `ThrottlingException` | `max_concurrency = 5` reached | Wait a moment and send again. |
| Predictions look wrong for a city | Coordinates do not match the city | Re-pick the city from the dropdown; it snaps the coordinates to that city's centre. |
| Dropdowns say "built-in defaults" | `preprocessing_artifacts.json` not readable | Run this notebook from the same directory as Notebook 03, or check S3 permissions on the `/processed` prefix. |
| Gradio UI does not appear | Studio/browser rendering issue | Check the printed local URL, or restart the notebook kernel and rerun. |
| Port 7860 already in use | An earlier demo is still running | Run the port-cleanup cell in Section 7. |
| Endpoint charges continue | Endpoint still exists | Serverless is ~$0 when idle, but delete it when the project is finished. |

For this tutorial, students should change:

```python
TEAM_ID = "teamXX"
STUDENT_ID = "sXXXX"
ENDPOINT_NAME = "iti113-teamXX-airbnb-price-slsv2"
```

## 10. Optional Cleanup Reminder

Do **not** run cleanup if the endpoint is still needed for demonstration.

A serverless endpoint scales to zero and bills per invocation, so an idle endpoint costs
approximately nothing — there is no urgency here. Delete it when the project is finished and it is
no longer being demonstrated.

In [ ]:
# Optional cleanup example. Uncomment only when you really want to delete the endpoint.

# endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
# endpoint_config_name = endpoint_desc["EndpointConfigName"]

# sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
# print("Deleted endpoint:", ENDPOINT_NAME)

# sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
# print("Deleted endpoint config:", endpoint_config_name)

print(f"Cleanup cell ready for endpoint: {ENDPOINT_NAME} (delete calls commented out)")